# TORAX Core Transport Optimization Check

Interactive verification check for optimizing tokamak core transport heating profiles using **BAUX Bayesian Optimization** on Google DeepMind's **TORAX 2.0.0** PDE simulator.

In [1]:
# 1. Environment & Setup
import os
import sys
from pathlib import Path

os.environ.setdefault("JAX_PRECISION", "f32")

# Ensure repository root is on sys.path for both local Jupyter and Colab
root = Path.cwd()
while root.name and not (root / "python" / "baux").exists():
    root = root.parent
if str(root / "python") not in sys.path:
    sys.path.insert(0, str(root / "python"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import torax
from baux import bo
from evals.torax.runner import ToraxRunner

print(f"TORAX version: {torax.__version__}")
print(f"BAUX loaded from: {root / 'python' / 'baux'}")

TORAX version: 2.0.0
BAUX loaded from: /Users/mehulbafna/research/baux/python/baux


## 2. Tokamak Physics & Core Transport Objective

We optimize auxiliary heating power $P_{\text{heat}} \in [10, 150]\text{ MW}$ and Gaussian radial deposition location $r/a \in [0.0, 0.8]$ to match central electron and ion temperature targets ($T_e = 15.0\text{ keV}$, $T_i = 12.0\text{ keV}$) with minimum total heating power.

In [2]:
# 2. Baseline Physics Simulation
runner = ToraxRunner(target_te=15.0, target_ti=12.0, max_steps=1)
print(f"Optimization bounds: {runner.bounds}")

# Run baseline at nominal midpoint (P_heat=80 MW, loc=0.4)
baseline_loss = runner.evaluate(p_heat_mw=80.0, gaussian_loc=0.4)
print(f"Baseline negative loss (P=80MW, loc=0.4): {baseline_loss:.4f}")

Optimization bounds: [[ 10.  150. ]
 [  0.    0.8]]
Baseline negative loss (P=80MW, loc=0.4): -11.3987


## 3. Matched Multi-Seed Optimization: BAUX vs. Random Baseline

Run a matched multi-seed comparison across fixed evaluation seeds to test sample efficiency and transport loss convergence.

In [3]:
# 3. Comparative Evaluation Loop
def run_baux_bo(runner, rounds=15, seed=42):
    rng = np.random.default_rng(seed)
    cfg = bo.OptimizerConfig()
    opt = bo.create_optimizer(bounds=runner.bounds, config=cfg, rng=rng)
    incumbents = []
    best_loss = -float("inf")
    for step in range(1, rounds + 1):
        pts = opt.ask(1)
        p_heat, loc = float(pts[0, 0]), float(pts[0, 1])
        score = runner.evaluate(p_heat, loc)
        opt.tell(pts, np.array([score]))
        if score > best_loss:
            best_loss = score
        incumbents.append(best_loss)
    return np.array(incumbents)

def run_random_search(runner, rounds=15, seed=42):
    rng = np.random.default_rng(seed)
    incumbents = []
    best_loss = -float("inf")
    b = runner.bounds
    for step in range(1, rounds + 1):
        p_heat = rng.uniform(b[0, 0], b[0, 1])
        loc = rng.uniform(b[1, 0], b[1, 1])
        score = runner.evaluate(p_heat, loc)
        if score > best_loss:
            best_loss = score
        incumbents.append(best_loss)
    return np.array(incumbents)

seeds = [17, 42, 101]
rounds = 10

print(f"Running matched comparison over seeds {seeds} ({rounds} rounds per seed)...")
baux_curves = [run_baux_bo(runner, rounds=rounds, seed=s) for s in seeds]
rand_curves = [run_random_search(runner, rounds=rounds, seed=s) for s in seeds]

baux_mean = np.mean(baux_curves, axis=0)
baux_sem = np.std(baux_curves, axis=0) / np.sqrt(len(seeds))

rand_mean = np.mean(rand_curves, axis=0)
rand_sem = np.std(rand_curves, axis=0) / np.sqrt(len(seeds))

print("Optimization complete.")
print(f"Final BAUX Best Score: {baux_mean[-1]:.3f} +/- {baux_sem[-1]:.3f}")
print(f"Final Random Best Score: {rand_mean[-1]:.3f} +/- {rand_sem[-1]:.3f}")

Running matched comparison over seeds [17, 42, 101] (10 rounds per seed)...
Optimization complete.
Final BAUX Best Score: -6.124 +/- 0.760
Final Random Best Score: -7.166 +/- 0.324


## 4. Telemetry & Convergence Visualization

In [4]:
# 4. Convergence Trajectory Plot
steps = np.arange(1, rounds + 1)
plt.figure(figsize=(8, 5), dpi=150)
plt.plot(steps, baux_mean, "o-", color="#1f77b4", label="BAUX Bayesian Search", lw=2)
plt.fill_between(steps, baux_mean - baux_sem, baux_mean + baux_sem, color="#1f77b4", alpha=0.2)

plt.plot(steps, rand_mean, "s--", color="#7f7f7f", label="Random Search", lw=2)
plt.fill_between(steps, rand_mean - rand_sem, rand_mean + rand_sem, color="#7f7f7f", alpha=0.2)

plt.axhline(baseline_loss, color="r", linestyle=":", label="Nominal Unoptimized Baseline")
plt.xlabel("Simulation Evaluations (Rounds)")
plt.ylabel("Negative Transport Loss (Higher is Better)")
plt.title("TORAX Core Transport Optimization: BAUX vs. Random Baseline")
plt.legend(frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()